In [1]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install -q sentence-transformers FlagEmbedding

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.1 MB/s eta 0:00:00


In [3]:
# ── 셀 3: chunks 업로드 ──────────────────────────────────────
# chunks_jo.json — 조 단위 (Hierarchical RAG fetch용)
# chunks_ho.json — 호 단위 (임베딩 & 벡터 검색 대상)
import os
for fname in ['chunks_jo.json', 'chunks_ho.json']:
    if os.path.exists(fname):
        os.remove(fname)
        print(f'기존 파일 삭제: {fname}')

from google.colab import files
uploaded = files.upload()  # chunks_jo.json, chunks_ho.json 두 파일 선택


Saving chunks_ho.json to chunks_ho.json
Saving chunks_jo.json to chunks_jo.json


In [4]:
import json
with open('chunks_ho.json', encoding='utf-8') as f:
    chunks = json.load(f)
print(f'ho 청크 수: {len(chunks)}')
print(chunks[0])


ho 청크 수: 6297
{'chunk_id': 'LCA_1', 'law_name': '지방계약법', 'article_id': '제1조', 'article_number': '제1조', 'text': '제1조(목적) 이 법은 지방자치단체를 당사자로 하는 계약에 관한 기본적인 사항을 정함으로써 계약업무를 원활하게 수행할 수 있도록 함을 목적으로 한다. [전문개정 2009. 2. 6.]', 'is_parent': False, 'parent_id': None, 'is_ref_article': False, 'is_upper_law': False, 'hierarchy': {'조': '제1조'}}


In [5]:
# ── 셀 4: 청크 로드 ──────────────────────────────────────────
# chunks_ho.json — 전체 청크 (parent + child, Hierarchical RAG 구조 유지)
# chunks_jo.json — 조 단위 parent 청크 (Hierarchical RAG fetch용)
import json

# ho 전체 로드
with open('chunks_ho.json', encoding='utf-8') as f:
    ho_chunks = json.load(f)

seen = {}
for c in ho_chunks:
    seen[c['chunk_id']] = c
ho_chunks = list(seen.values())

# 조 단위 로드
with open('chunks_jo.json', encoding='utf-8') as f:
    jo_chunks = json.load(f)

seen_jo = {}
for c in jo_chunks:
    seen_jo[c['chunk_id']] = c
jo_chunks = list(seen_jo.values())

# 임베딩/검색 대상: parent 제외 child만
# parent는 Hierarchical RAG fetch용이므로 검색 풀에서 제외
search_chunks = [c for c in ho_chunks if not c.get('is_parent', False)]
texts     = [c['text']     for c in search_chunks]
chunk_ids = [c['chunk_id'] for c in search_chunks]

# jo 임베딩 대상
jo_texts     = [c['text']     for c in jo_chunks]
jo_chunk_ids = [c['chunk_id'] for c in jo_chunks]

print(f'전체 ho 청크:            {len(ho_chunks)}개')
print(f'parent 청크:             {sum(1 for c in ho_chunks if c.get("is_parent"))}개')
print(f'임베딩 대상 (child만):   {len(search_chunks)}개')
print(f'jo 청크 (fetch용):       {len(jo_chunks)}개')


전체 ho 청크:            6297개
parent 청크:             834개
임베딩 대상 (child만):   5463개
jo 청크 (fetch용):       1422개


In [6]:
# ── 셀 5: 평가셋 생성 ────────────────────────────────────────
# ho 청크 기반 평가셋 + jo 청크 기반 평가셋 각각 생성
import random

def make_eval_dataset(chunks, n=200):
    templates = [
        '다음 내용은 어떤 규정을 설명하나요?',
        '이 조항은 무엇에 관한 내용인가요?',
        '다음 규정과 관련된 조문을 찾아주세요.',
        '계약 실무에서 아래 내용은 어떤 법적 근거에 해당하나요?'
    ]
    eval_data = []
    qid = 1
    for chunk in chunks:
        text = chunk['text']
        if len(text) < 80:
            continue
        query = random.choice(templates) + '\n\n' + text[:120]
        eval_data.append({
            'query_id':      f'eval_{qid:03d}',
            'query':         query,
            'relevant_docs': [chunk['chunk_id']]
        })
        qid += 1
    random.shuffle(eval_data)
    return eval_data[:n]

# ho 평가셋 — 호 단위 검색 성능 측정
eval_data_ho = make_eval_dataset(search_chunks, n=200)
with open('eval_dataset_ho.json', 'w', encoding='utf-8') as f:
    json.dump(eval_data_ho, f, ensure_ascii=False, indent=2)
print(f'ho 평가셋: {len(eval_data_ho)}개')

# jo 평가셋 — 조 단위 검색 성능 측정
eval_data_jo = make_eval_dataset(jo_chunks, n=200)
with open('eval_dataset_jo.json', 'w', encoding='utf-8') as f:
    json.dump(eval_data_jo, f, ensure_ascii=False, indent=2)
print(f'jo 평가셋: {len(eval_data_jo)}개')


ho 평가셋: 200개
jo 평가셋: 200개


In [7]:
# ── 셀 6: 모델 평가 함수 ─────────────────────────────────────
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_model(model_name, texts, chunk_ids, eval_data, batch_size=8):
    """
    인자:
        texts      — 임베딩할 청크 텍스트 리스트
        chunk_ids  — 청크 ID 리스트 (texts와 순서 동일)
        eval_data  — 평가셋 (query_id / query / relevant_docs)
    """
    print(f'\n===== {model_name} =====')

    model = SentenceTransformer(model_name, device='cuda')

    doc_vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    recall1 = recall5 = recall10 = mrr = 0

    for item in eval_data:
        gt_docs = set(item['relevant_docs'])
        qvec = model.encode(
            item['query'],
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        scores     = cosine_similarity([qvec], doc_vectors)[0]
        ranked_ids = [chunk_ids[i] for i in np.argsort(scores)[::-1]]

        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    n = len(eval_data)
    del doc_vectors, model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'Model':     model_name,
        'Recall@1':  round(recall1  / n, 4),
        'Recall@5':  round(recall5  / n, 4),
        'Recall@10': round(recall10 / n, 4),
        'MRR':       round(mrr      / n, 4),
    }


In [8]:
# ── 셀 7: BGE-M3 평가 ────────────────────────────────────────
import pandas as pd

print("▶ ho (호 단위) 평가")
result_ho = evaluate_model('BAAI/bge-m3', texts, chunk_ids, eval_data_ho, batch_size=8)
display(pd.DataFrame([result_ho]))

print("\n▶ jo (조 단위) 평가")
result_jo = evaluate_model('BAAI/bge-m3', jo_texts, jo_chunk_ids, eval_data_jo, batch_size=8)
display(pd.DataFrame([result_jo]))


▶ ho (호 단위) 평가

===== BAAI/bge-m3 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

,Model,Recall@1,Recall@5,Recall@10,MRR
0,BAAI/bge-m3,0.835,0.965,1.0,0.889



▶ jo (조 단위) 평가

===== BAAI/bge-m3 =====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/178 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,BAAI/bge-m3,0.995,1.0,1.0,0.9975


In [9]:
# ── 셀 8: KURE-v1 평가 ───────────────────────────────────────
print("▶ ho (호 단위) 평가")
result_ho = evaluate_model('nlpai-lab/KURE-v1', texts, chunk_ids, eval_data_ho, batch_size=8)
display(pd.DataFrame([result_ho]))

print("\n▶ jo (조 단위) 평가")
result_jo = evaluate_model('nlpai-lab/KURE-v1', jo_texts, jo_chunk_ids, eval_data_jo, batch_size=8)
display(pd.DataFrame([result_jo]))


▶ ho (호 단위) 평가

===== nlpai-lab/KURE-v1 =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

,Model,Recall@1,Recall@5,Recall@10,MRR
0,nlpai-lab/KURE-v1,0.825,0.975,1.0,0.8869



▶ jo (조 단위) 평가

===== nlpai-lab/KURE-v1 =====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/178 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,nlpai-lab/KURE-v1,0.995,1.0,1.0,0.9975


In [10]:
# ── 셀 9: Qwen3 평가 ─────────────────────────────────────────
print("▶ ho (호 단위) 평가")
result_ho = evaluate_model('Qwen/Qwen3-Embedding-0.6B', texts, chunk_ids, eval_data_ho, batch_size=1)
display(pd.DataFrame([result_ho]))

print("\n▶ jo (조 단위) 평가")
result_jo = evaluate_model('Qwen/Qwen3-Embedding-0.6B', jo_texts, jo_chunk_ids, eval_data_jo, batch_size=1)
display(pd.DataFrame([result_jo]))


▶ ho (호 단위) 평가

===== Qwen/Qwen3-Embedding-0.6B =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

,Model,Recall@1,Recall@5,Recall@10,MRR
0,Qwen/Qwen3-Embedding-0.6B,0.83,0.96,0.995,0.8878



▶ jo (조 단위) 평가

===== Qwen/Qwen3-Embedding-0.6B =====


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/1422 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,Qwen/Qwen3-Embedding-0.6B,0.975,1.0,1.0,0.9867


In [11]:
# ── 셀 10: BGE-M3 dense + sparse 임베딩 및 저장 ──────────────
# ho 컬렉션 (검색 대상), jo 컬렉션 (Hierarchical RAG fetch용) 각각 임베딩
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import json

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

# ── ho 임베딩 ──────────────────────────────────────────────────
print(f'ho 임베딩 대상: {len(search_chunks)}개')
output_ho = model.encode(
    texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
    batch_size=8,
)

np.savez(
    'vectors_ho.npz',
    vectors=output_ho['dense_vecs'],
    chunk_ids=np.array(chunk_ids)
)

sparse_ho = [{k: float(v) for k, v in sw.items()} for sw in output_ho['lexical_weights']]
with open('sparse_weights_ho.json', 'w', encoding='utf-8') as f:
    json.dump(sparse_ho, f)

print(f'✅ ho 저장 완료! dense shape: {output_ho["dense_vecs"].shape}')

# ── jo 임베딩 ──────────────────────────────────────────────────
print(f'\njo 임베딩 대상: {len(jo_chunks)}개')
output_jo = model.encode(
    jo_texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
    batch_size=8,
)

np.savez(
    'vectors_jo.npz',
    vectors=output_jo['dense_vecs'],
    chunk_ids=np.array(jo_chunk_ids)
)

sparse_jo = [{k: float(v) for k, v in sw.items()} for sw in output_jo['lexical_weights']]
with open('sparse_weights_jo.json', 'w', encoding='utf-8') as f:
    json.dump(sparse_jo, f)

print(f'✅ jo 저장 완료! dense shape: {output_jo["dense_vecs"].shape}')

# 다운로드
from google.colab import files
files.download('vectors_ho.npz')
files.download('sparse_weights_ho.json')
files.download('vectors_jo.npz')
files.download('sparse_weights_jo.json')


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ho 임베딩 대상: 5463개


Inference Embeddings: 100%|██████████| 683/683 [00:18<00:00, 36.50it/s]


✅ ho 저장 완료! dense shape: (5463, 1024)

jo 임베딩 대상: 1422개


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>